# Notebook B — controlled legacy-development workflow

Historical saved outputs are preserved verbatim and predate this edited source. They are not a new execution or corrected research evidence. Notebook B archival outputs identify f538726; the readiness base is f66bb57.

Run in a fresh kernel. Explicitly acknowledge legacy data/checkpoint semantics. Use a fresh OUTPUT_DIR and the identical code, data, checkpoint, units and split in A–D. A → B → C/D; C requires A and B, D requires B. Scientific projected training uses the separately documented audited baseline. No CAMELS transfer claims from legacy GroupMass.


In [ ]:
# Settings — edit these before Run All.
import os
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/cosmic-net")
REPO_URL = "https://github.com/Neal-Salian/cosmic-net-f.git"
REPO_BRANCH = "codex/fix-stageb-budget-feasibility"
SYNC_REPO = True                 # False for an already prepared local checkout.
INSTALL_MISSING = True
INPUT_DIR = None                 # None: discover one matching Kaggle dataset.
INPUT_ROOT = Path("/kaggle/input")
DEVICE = "auto"                  # "auto", "cpu", or "cuda"
SMOKE_TEST = False
SMOKE_GRAPHS_PER_SPLIT = 3
GRAPH_BACKEND = "auto"           # "auto" or "torch" (no native extensions)
OUTPUT_DIR = None  # Default: outputs/rls/notebook_B
POLICY_EPOCHS = None  # None uses config.yaml; smoke runs cap epochs at two.
POLICY_OVERRIDES = {}  # Training/reward overrides, recorded for C and D.
RESUME_POLICY_DIR = None  # Verified training_checkpoint.json folder; skips policy training.
RUN_STAGE_B = True
STAGEB_EPOCHS = None
RUN_PENALTY_ABLATION = False

ACKNOWLEDGE_LEGACY_INPUTS = False  # Required for a non-smoke historical CSV/checkpoint campaign.
EXPECTED_REPO_COMMIT = None       # Set to the exact verified commit before the campaign.
LEGACY_CSV_UNITS = {"position": "ckpc_per_h", "radius": "ckpc_per_h", "stellar_mass": "log10_Msun", "halo_mass": "M_sun"}
FRACTIONS = [0.4, 0.6, 0.8, 1.0]

# Explicit settings for this new private campaign.
EXPECTED_REPO_COMMIT = 'c825bf57652005c80faff79cf4dfd58e3a9baa14'
ACKNOWLEDGE_LEGACY_INPUTS = True
SYNC_REPO = False
DEVICE = "cuda"
INPUT_ROOT = Path("/kaggle/input")
_input_candidates = [Path("/kaggle/input/datasets/nealsalian/cosmicnet-data"),
                     Path("/kaggle/input/cosmicnet-data")]
INPUT_DIR = next((path for path in _input_candidates if (path / "tng100_clustered.csv").is_file()), None)
if INPUT_DIR is None:
    raise FileNotFoundError("Pinned cosmicnet-data inputs are not mounted")
OUTPUT_DIR = Path("/kaggle/working/cosmic-net-runs/notebook_B")
RESUME_POLICY_DIR = Path("/kaggle/working/recovered-training")


In [ ]:
# Dependencies — retain Kaggle's PyTorch and install only missing core packages.
import importlib
import importlib.util
import subprocess
import torch

print("Python:", sys.version.split()[0], "| torch:", torch.__version__,
      "| CUDA build:", torch.version.cuda, "| GPU available:", torch.cuda.is_available())
packages = {"torch_geometric": "torch-geometric>=2.6,<3", "yaml": "pyyaml",
            "numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
            "matplotlib": "matplotlib", "h5py": "h5py", "requests": "requests", "dotenv": "python-dotenv"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    if not INSTALL_MISSING:
        raise RuntimeError("Missing packages: " + ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
    importlib.invalidate_caches()
# NNConv works without torch-scatter/torch-sparse. Do not install CPU extension
# wheels into a CUDA runtime or compile unsupported wheels during notebook setup.
import torch_geometric
print("PyG:", torch_geometric.__version__)


In [ ]:
# Verified local Git bundle delivery; no GitHub branch push.
import base64, hashlib, subprocess
_delivery = base64.b64decode('...')
assert hashlib.sha256(_delivery).hexdigest() == '88043f8cfcf484907989a6c78608add7b70a2c576a974870b4b619e72f08c47b'
_bundle_path = Path("/kaggle/working/cosmic-net-stageb-readiness.bundle")
_bundle_path.write_bytes(_delivery)
_git_env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
if REPO_DIR.exists():
    raise FileExistsError("Use a fresh kernel; refusing to overwrite an existing checkout")
subprocess.run(["git", "clone", "--depth", "1", "--no-checkout", "--single-branch",
    "--branch", "fix/research-pipeline-readiness", REPO_URL, str(REPO_DIR)], check=True, env=_git_env)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--no-auto-gc", str(_bundle_path),
    "refs/heads/codex/fix-stageb-budget-feasibility"], check=True, env=_git_env)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", 'c825bf57652005c80faff79cf4dfd58e3a9baa14'], check=True, env=_git_env)
REPO_HEAD = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
assert REPO_HEAD == EXPECTED_REPO_COMMIT == 'c825bf57652005c80faff79cf4dfd58e3a9baa14'
sys.path.insert(0, str(REPO_DIR))
print("Verified evaluator commit:", REPO_HEAD, "| delivery: private local Git bundle")

import io, zipfile
_recovery_bytes = base64.b64decode('...')
assert hashlib.sha256(_recovery_bytes).hexdigest() == '973cd899af2450495aba10e9b1397819a18d74c83389afd19c804711cb6f199a'
_recovery_dir = Path(RESUME_POLICY_DIR)
_recovery_dir.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(_recovery_bytes)) as archive:
    for name in archive.namelist():
        if Path(name).name != name or name not in {"policy.pt", "value_net.pt", "training_log.csv", "random_validation.csv", "policy_validation.json", "training_checkpoint.json"}:
            raise ValueError("Unexpected recovery archive entry")
        (_recovery_dir / name).write_bytes(archive.read(name))
print("Recovered policy SHA256:", hashlib.sha256((_recovery_dir / "policy.pt").read_bytes()).hexdigest())


In [ ]:
# Locate and validate the attached dataset before any model or graph work.
import numpy as np
import pandas as pd

def locate_inputs(input_dir, input_root):
    if input_dir is not None:
        candidates = [Path(input_dir)]
    else:
        candidates = sorted({p.parent for p in Path(input_root).rglob("tng100_clustered.csv")
                             if (p.parent / "best_model_augmented.pt").is_file()})
    if len(candidates) != 1:
        raise FileNotFoundError("Set INPUT_DIR to the dataset folder containing "
                                "tng100_clustered.csv and best_model_augmented.pt. "
                                f"Matching folders: {candidates}")
    folder = candidates[0].resolve()
    csv_path = folder / "tng100_clustered.csv"
    checkpoint_path = folder / "best_model_augmented.pt"
    for path in (csv_path, checkpoint_path):
        if not path.is_file() or path.stat().st_size == 0:
            raise FileNotFoundError(f"Missing or empty input: {path}")
        with path.open("rb") as stream:
            if stream.read(128).startswith(b"version https://git-lfs.github.com/spec/v1"):
                raise ValueError(f"{path} is a Git LFS pointer, not the actual data file.")
    return csv_path, checkpoint_path

CSV_PATH, CHECKPOINT_PATH = locate_inputs(INPUT_DIR, INPUT_ROOT)
catalog = pd.read_csv(CSV_PATH)
velocity_column = "vel_dispersion" if "vel_dispersion" in catalog else "velocity_dispersion"
required = ["subhalo_id", "group_id", "stellar_mass", velocity_column,
            "half_mass_radius", "metallicity", "pos_x", "pos_y", "pos_z",
            "vel_x", "vel_y", "vel_z"]
target_column = "halo_mass" if "halo_mass" in catalog else "halo_mass_log"
required.append(target_column)
missing_columns = sorted(set(required) - set(catalog.columns))
if missing_columns:
    raise ValueError(f"CSV missing required columns: {missing_columns}")
numeric = catalog[required].apply(pd.to_numeric, errors="coerce")
if catalog.empty or not np.isfinite(numeric.to_numpy()).all():
    raise ValueError("CSV must contain finite numeric features and targets in every row.")
if (numeric[[velocity_column, "half_mass_radius", "metallicity"]] < 0).any().any():
    raise ValueError("Velocity dispersion, radius and metallicity cannot be negative.")
print("CSV:", CSV_PATH, "| rows:", len(catalog), "| halos:", catalog.group_id.nunique())
print("Checkpoint:", CHECKPOINT_PATH, "| bytes:", CHECKPOINT_PATH.stat().st_size)


In [ ]:
# Configuration and imports — retain the repository's RL settings.
import copy
import yaml
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader
from data.loaders.base_loader import get_loader
from graph.graph_builder import GraphBuilder
from model.model import build_model
from model.physics_loss import MetricsComputer
from rls.provenance import record_backbone, require_backbone_label

_B_IMPORTED_HEAD = REPO_HEAD
cfg = yaml.safe_load((REPO_DIR / "config/config.yaml").read_text())
cfg["data"].update(source="tng", num_workers=0, batch_size=16)
cfg["data"]["tng"]["clustered_file"] = str(CSV_PATH)
cfg["data"]["tng"]["clustered_units"] = copy.deepcopy(LEGACY_CSV_UNITS)
cfg["data"]["checkpoint_research_label"] = "legacy_total_radius"
cfg["rls"].update(evaluation_decoder="direct", evaluation_constraint="no_isolates", edge_feature_names=cfg["graph"]["edge_features"])
if not SMOKE_TEST and not ACKNOWLEDGE_LEGACY_INPUTS:
    raise ValueError("Explicit ACKNOWLEDGE_LEGACY_INPUTS=True is required; these inputs do not certify corrected astronomy research.")
if DEVICE not in ("auto", "cpu", "cuda"):
    raise ValueError("DEVICE must be auto, cpu or cuda.")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA requested but unavailable. Enable a Kaggle GPU or use DEVICE='cpu'.")
device = torch.device("cuda" if DEVICE == "cuda" or
                      (DEVICE == "auto" and torch.cuda.is_available()) else "cpu")
if GRAPH_BACKEND not in ("auto", "torch"):
    raise ValueError("GRAPH_BACKEND must be auto or torch.")
if SMOKE_GRAPHS_PER_SPLIT < 2:
    raise ValueError("Use at least two smoke graphs per split.")
from rls.notebook_workflow import fresh_output_directory
OUT = Path(OUTPUT_DIR) if OUTPUT_DIR is not None else Path("/kaggle/working/cosmic-net-runs/notebook_B")
if SMOKE_TEST:
    OUT = OUT / "smoke"
OUT = fresh_output_directory(OUT)
cfg["data"]["tng"]["cache_dir"] = str(OUT / "tng_cache")
import json
(OUT / "provenance.json").write_text(json.dumps({"B": {"stage": "B", "status": "running", "repo": {"head": REPO_HEAD}}}, indent=2))
seed = int(cfg.get("seed", 42))
torch.manual_seed(seed)
np.random.seed(seed)
print("Device:", device, "| run mode:", "SMOKE" if SMOKE_TEST else "FULL", "| outputs:", OUT)


In [ ]:
# Load only the supplied checkpoint; use its own architecture and graph settings.
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
checkpoint_config = checkpoint.get("config") if isinstance(checkpoint, dict) else None
if checkpoint_config and "model" in checkpoint_config:
    cfg["model"] = copy.deepcopy(checkpoint_config["model"])
    if "graph" in checkpoint_config:
        cfg["graph"] = copy.deepcopy(checkpoint_config["graph"])
cfg["model"]["mc_samples"] = 30
if cfg["graph"].get("hierarchical", False):
    raise ValueError("Notebook B supports single-level graphs; hierarchical mode is only a scaffold.")
from model.model import load_model
# Legacy metadata never upgrades these weights to projected/audited artifacts.
if checkpoint.get("artifact_type") == "projected_backbone":
    raise ValueError("Use the audited projected workflow; this notebook expects the explicit legacy schema.")
gnn = load_model(str(CHECKPOINT_PATH), cfg, device, research_label="legacy_total_radius")
gnn = gnn.to(device).eval()
gnn.requires_grad_(False)  # Saliency still differentiates input features.
model_device = next(gnn.parameters()).device
backbone_record = record_backbone("frozen", gnn)
del checkpoint
print("Loaded:", CHECKPOINT_PATH.name, "| dimensions:", gnn.hidden_dim, gnn.output_dim,
      "| parameters:", sum(p.numel() for p in gnn.parameters()))

cfg["rls"]["edge_feature_names"] = list(cfg["graph"]["edge_features"])


In [ ]:
# Build each halo independently; fail on invalid graphs instead of dropping them.
def validate_graph(graph):
    if graph.num_nodes < 1 or graph.edge_index.shape[1] == 0:
        raise ValueError(f"Empty graph or no edges: {graph.cluster_id}")
    graph.validate(raise_on_error=True)
    if graph.x.shape[1] != gnn.node_input_dim or graph.edge_attr.shape != (
            graph.edge_index.shape[1], gnn.edge_input_dim):
        raise ValueError(f"Feature dimensions incompatible with checkpoint: {graph.cluster_id}")
    for name in ("x", "pos", "edge_attr", "y"):
        value = getattr(graph, name, None)
        if value is None or not torch.isfinite(value).all():
            raise ValueError(f"Missing or non-finite {name}: {graph.cluster_id}")
    if graph.pos.shape != (graph.num_nodes, 3):
        raise ValueError(f"Expected physical 3D positions: {graph.cluster_id}")

loader = get_loader(cfg)
halos = loader.load()
split_halos = dict(zip(("train", "val", "test"), loader.split_data(halos)))
full_split_sizes = {name: len(items) for name, items in split_halos.items()}
if any(size < 2 for size in full_split_sizes.values()):
    raise ValueError(f"Each split needs at least two halos; got {full_split_sizes}.")
if SMOKE_TEST:
    split_halos = {name: items[:SMOKE_GRAPHS_PER_SPLIT] for name, items in split_halos.items()}
torch.manual_seed(seed)
np.random.seed(seed)
builder = GraphBuilder(cfg)
builder.backend_used = "portable_torch"
builder.backend_used = "portable_torch"
import time
graph_start = time.perf_counter()
import time
graph_start = time.perf_counter()
split_graphs = {}
for name, items in split_halos.items():
    graphs = []
    for halo in items:
        graph = builder.build_graph(halo)
        validate_graph(graph)
        graphs.append(graph)
    split_graphs[name] = graphs
train_loader = DataLoader(split_graphs["train"], batch_size=cfg["data"]["batch_size"], shuffle=True)
val_loader = DataLoader(split_graphs["val"], batch_size=cfg["data"]["batch_size"], shuffle=False)
test_loader = DataLoader(split_graphs["test"], batch_size=cfg["data"]["batch_size"], shuffle=False)
print("Full split:", full_split_sizes, "| evaluated split:",
      {name: len(graphs) for name, graphs in split_graphs.items()},
      "| graph backend: portable torch | tie policy: inclusive")
graph_construction_time_s = time.perf_counter() - graph_start

graph_construction_time_s = time.perf_counter() - graph_start


In [ ]:
from rls.notebook_workflow import (
    run_context, file_info, checked_artifact, source_artifacts, load_saved_policy,
    prepared_splits, predict_graph, prediction_pair, policy_masks, evaluate_masks,
    result_rows, merge_baselines, physics_diagnostics, coverage_rows,
    adaptation_trial, validation_gates, complete_stage, local_camels_graphs,
    policy_metrics, random_pair_reference, policy_validation)
from rls.evaluate import save_paper_plots
from rls.policy import build_policy
from rls.policy_gradient import ValueNet, PolicyGradientTrainer
from rls.train_policy import train_policy
from rls.stageb import fine_tune_gnn
context = run_context(cfg, CSV_PATH, CHECKPOINT_PATH, split_graphs, backbone_record, SMOKE_TEST)
prepared = prepared_splits(split_graphs, gnn, device)
train_graphs, val_graphs, test_graphs = (prepared[name] for name in ("train", "val", "test"))
artifacts = []
dependencies = {}
run_mode = context["run_mode"]
g = test_graphs[0]
full_mask = torch.ones(g["edge_index"].shape[1], dtype=torch.bool, device=device)
print("=== DEVICE CHECK ===")
print("GNN:", next(gnn.parameters()).device)
for key in ("x", "edge_index", "edge_attr", "emb", "ctx"):
    print(key + ":", g[key].device)
print("Single-halo full-graph prediction:", float(predict_graph(gnn, g, full_mask)))

context["graph_construction_time_s"] = graph_construction_time_s

from rls.notebook_budget import budget_cohort, require_budget_cohort, record_budget_realizations


In [ ]:
cfg["rls"]["sparsity_mode"] = "pair_pl"
cfg["rls"]["policy_normalize"] = True
cfg["rls"].update(POLICY_OVERRIDES)
if cfg["rls"]["sparsity_mode"] != "pair_pl":
    raise ValueError("Notebook B requires the repaired pair_pl objective.")
policy_epochs = cfg["rls"]["epochs"] if POLICY_EPOCHS is None else POLICY_EPOCHS
stageb_epochs = cfg["rls"].get("stageb_epochs", 10) if STAGEB_EPOCHS is None else STAGEB_EPOCHS
if any(not isinstance(value, int) or isinstance(value, bool) or value < 1 for value in (policy_epochs, stageb_epochs)):
    raise ValueError("Policy and Stage B epochs must be positive integers.")
if SMOKE_TEST:
    policy_epochs, stageb_epochs = min(policy_epochs, 2), min(stageb_epochs, 2)
cfg["rls"].update(epochs=policy_epochs, stageb_epochs=stageb_epochs)
rls_cfg = cfg["rls"]
final_training_keep = float(rls_cfg["target_sparsity_end"])
# Exact-mask cohorts are topology-only; the original split manifest remains intact.
budget_graphs, budget_reports = {}, {}
for split_name, original_graphs in prepared.items():
    budget_graphs[split_name], budget_reports[split_name] = budget_cohort(
        original_graphs, rls_cfg, split=split_name, target_sparsity=final_training_keep)
(OUT / "budget_eligibility.json").write_text(json.dumps(budget_reports, indent=2, allow_nan=False))
artifacts.append("budget_eligibility.json")
print("Exact-budget cohorts:", {name: {key: report[key] for key in
    ("original_count", "eligible_count", "excluded_count")} for name, report in budget_reports.items()})
require_budget_cohort(budget_reports["val"], minimum=2)
require_budget_cohort(budget_reports["test"], minimum=2)
validation_graphs = budget_graphs["val"]
# Stage A retains its declared legacy-repaired training population.
from rls.notebook_recovery import (save_training_checkpoint, load_training_checkpoint,
    ARTIFACT_NAMES, MANIFEST_NAME)
recovery_record = None
if RESUME_POLICY_DIR is not None:
    import shutil
    policy, training_cfg, recovery_record = load_training_checkpoint(
        RESUME_POLICY_DIR, context, cfg, gnn, device,
        allow_source_revision_change=True, evaluator_repo_head=REPO_HEAD,
        evaluation_campaign=str(OUT))
    cfg["rls"]["reward_error_scale"] = training_cfg["rls"]["reward_error_scale"]
    preserved_training = OUT / "recovered_training"
    preserved_training.mkdir(exist_ok=False)
    for filename in (*ARTIFACT_NAMES, MANIFEST_NAME):
        shutil.copy2(Path(RESUME_POLICY_DIR) / filename, preserved_training / filename)
        artifacts.append("recovered_training/" + filename)
    for filename in ("policy.pt", "value_net.pt", "training_log.csv"):
        shutil.copy2(preserved_training / filename, OUT / filename)
    selected_epoch = recovery_record["policy_validation"].get("selected_epoch")
    (OUT / "recovery_record.json").write_text(json.dumps(recovery_record, indent=2, allow_nan=False))
    artifacts.append("recovery_record.json")
    print("Recovered completed policy; policy training is skipped. Training source:",
          recovery_record["training_source"])
else:
    policy = build_policy(cfg, node_emb_dim=gnn.output_dim).to(device)
    value_net = ValueNet(gnn.output_dim).to(device)
    trainer = PolicyGradientTrainer(policy, value_net,
        torch.optim.Adam(policy.parameters(), lr=rls_cfg["lr"]),
        torch.optim.Adam(value_net.parameters(), lr=rls_cfg["lr"]), rls_cfg)
random_val_rows = random_pair_reference(validation_graphs, gnn, rls_cfg,
    range(2 if SMOKE_TEST else int(rls_cfg.get("random_validation_seeds", 10))), target_sparsity=final_training_keep)
pd.DataFrame(random_val_rows).to_csv(OUT / "random_validation.csv", index=False)
log_rows = []
def training_log(epoch, target, loss):
    if not np.isfinite(loss):
        raise RuntimeError(f"Nonfinite training loss at epoch {epoch}; no completed policy will be published.")
    log_rows.append(dict(epoch=epoch, target_sp=target, loss=loss))
    print(f"Epoch {epoch + 1}/{policy_epochs}: target={target:.3f}, loss={loss:.4f}")
if RESUME_POLICY_DIR is None:
    losses = train_policy(trainer, train_graphs, prediction_pair(gnn), rls_cfg,
                          device, epochs=policy_epochs, log_fn=training_log,
                          validation_fn=lambda p: policy_metrics(p, validation_graphs, gnn, rls_cfg, target_sparsity=final_training_keep))
    selected_epoch = trainer.selected_epoch
policy.eval()
selected_val = policy_metrics(policy, validation_graphs, gnn, rls_cfg, target_sparsity=final_training_keep)
policy_quality = policy_validation(selected_val, random_val_rows)
policy_quality["selected_epoch"] = selected_epoch
policy_quality["scope"] = "smoke" if SMOKE_TEST else "validation"
policy_quality["execution_cohort"] = copy.deepcopy(budget_reports["val"])
policy_filename = "policy.pt"  # Preserve valid negative results for downstream comparison.
selected_val["run_mode"] = run_mode
for row in random_val_rows:
    row["run_mode"] = run_mode
if RESUME_POLICY_DIR is None:
    torch.save(policy.state_dict(), OUT / policy_filename)
print("Policy validation:", policy_quality)
if RESUME_POLICY_DIR is None:
    torch.save(value_net.state_dict(), OUT / "value_net.pt")
    pd.DataFrame(trainer.diagnostics).to_csv(OUT / "training_log.csv", index=False)
artifacts += [policy_filename, "value_net.pt", "training_log.csv", "random_validation.csv"]
(OUT / "policy_validation.json").write_text(json.dumps(policy_quality, indent=2))
artifacts.append("policy_validation.json")

if RESUME_POLICY_DIR is None:
    save_training_checkpoint(OUT, context, cfg, policy_quality, REPO_HEAD)
    artifacts.append(MANIFEST_NAME)


In [ ]:
stageb_model, stageb_record = None, None
stageb_info = {"enabled": RUN_STAGE_B, "requested_keep_fraction": final_training_keep,
    "training_graph_ids": budget_reports["train"]["eligible_graph_ids"],
    "validation_graph_ids": budget_reports["val"]["eligible_graph_ids"],
    "excluded_training_graph_ids": budget_reports["train"]["excluded_graph_ids"]}
if (RUN_STAGE_B and (policy_quality["verdict"] == "PASS" or SMOKE_TEST)
        and budget_reports["train"]["eligible_count"] > 0):
    train_masks, _ = policy_masks(policy, budget_graphs["train"], rls_cfg, target_sparsity=final_training_keep)
    val_masks, _ = policy_masks(policy, validation_graphs, rls_cfg, target_sparsity=final_training_keep)
    record_budget_realizations(budget_reports["train"], budget_graphs["train"], train_masks)
    record_budget_realizations(budget_reports["val"], validation_graphs, val_masks)
    stageb_model = copy.deepcopy(gnn).requires_grad_(True)
    history, info = fine_tune_gnn(stageb_model, budget_graphs["train"], train_masks,
        epochs=stageb_epochs, lr=float(rls_cfg.get("stageb_lr", 1e-4)), device=device,
        val_graphs=validation_graphs, val_masks=val_masks,
        patience=int(rls_cfg.get("stageb_patience", 3)),
        full_tol=float(rls_cfg.get("stageb_full_tol", 0.02)),
        full_loss_weight=float(rls_cfg.get("stageb_full_loss_weight", 0.5)))
    if not np.isfinite(history).all():
        raise RuntimeError("Stage B produced nonfinite losses.")
    # No admissible validation epoch: restore the original model.
    if info["best_epoch"] < 0:
        stageb_model.load_state_dict(gnn.state_dict())
    stageb_model.eval().requires_grad_(False)
    stageb_record = record_backbone("stageB_finetuned" if info["accepted"] else "frozen", stageb_model)
    stageb_info.update(info, restored_frozen=info["best_epoch"] < 0)
    if info["accepted"]:
        torch.save(stageb_model.state_dict(), OUT / "finetuned_gnn.pt")
        artifacts.append("finetuned_gnn.pt")
    else:
        print("No accepted fine-tuned artifact is written.")
        print("No accepted Stage B model; original frozen backbone restored.")
    pd.DataFrame({"epoch": range(len(history)), "loss": history,
                  "val_full_rmse": info["full_hist"], "val_pruned_rmse": info["pruned_hist"]}).to_csv(OUT / "stageb_training_log.csv", index=False)
    artifacts += ["stageb_training_log.csv"]
    print("Stage B:", stageb_info)
else:
    print("No accepted fine-tuned artifact is written.")
    stageb_info["skip_reason"] = ("disabled" if not RUN_STAGE_B else
        "no_exact_feasible_training_graphs" if not budget_reports["train"]["eligible_count"] else
        "policy_validation_failed")
    print("Stage B skipped:", stageb_info["skip_reason"])


In [ ]:
evaluation_graphs = budget_graphs["test"]
test_masks, _ = policy_masks(policy, evaluation_graphs, rls_cfg, target_sparsity=final_training_keep)
record_budget_realizations(budget_reports["test"], evaluation_graphs, test_masks)
results = evaluate_masks(gnn, evaluation_graphs, test_masks)
rows = result_rows(results, backbone_record, run_mode)
if stageb_model is not None and stageb_info.get("accepted", False):
    stageb_rows = result_rows(evaluate_masks(stageb_model, evaluation_graphs, test_masks), stageb_record, run_mode)
    for row in stageb_rows:
        row["method"] += "_stageB"
    rows += stageb_rows
# Full and pruned rows use the same topology-defined test population.
physical_keep = float(np.mean([r["realized_pair_count"] / r["available_pair_count"]
    for r in budget_reports["test"]["graphs"] if r["eligible"]]))
for row in rows:
    row["mean_stored_keep_frac"] = row["mean_keep_frac"]
    row["mean_keep_frac"] = row["keep_frac"] = 1.0 if row["method"].startswith("full") else physical_keep
    row.update(comparison_population="topology_only_exact_budget_feasible",
        evaluation_graph_ids=budget_reports["test"]["eligible_graph_ids"],
        n_requested=budget_reports["test"]["original_count"],
        n_evaluated=budget_reports["test"]["eligible_count"],
        n_excluded=budget_reports["test"]["excluded_count"],
        comparison_requested_keep_fraction=final_training_keep,
        retention_units="distinct_nonself_physical_pairs")
results_df = pd.DataFrame(rows)
results_df.to_csv(OUT / "policy_results.csv", index=False)
artifacts.append("policy_results.csv")
print(results_df.to_string(index=False))

validation_report = {"full": result_rows(evaluate_masks(gnn, validation_graphs, policy_masks(policy, validation_graphs, rls_cfg, target_sparsity=final_training_keep)[0]), backbone_record, run_mode)[0],
                     "policy": selected_val, "quality": policy_quality}
(OUT / "validation_report.json").write_text(json.dumps(validation_report, indent=2))
artifacts.append("validation_report.json")

record_budget_realizations(budget_reports["test"], evaluation_graphs, test_masks)
(OUT / "budget_eligibility.json").write_text(json.dumps(budget_reports, indent=2, allow_nan=False))


In [ ]:
if RUN_PENALTY_ABLATION:
    raise ValueError("Legacy Bernoulli ablation is not supported by the repaired Notebook B workflow.")
print("Legacy penalty ablation disabled.")


In [ ]:
assert record_backbone("frozen", gnn) == backbone_record, "Frozen backbone changed during this run."
context["config_used"] = copy.deepcopy(cfg)
manifest = complete_stage(OUT, "B", context, artifacts,
    repo={"head": REPO_HEAD, "branch": REPO_BRANCH}, graph_backend=builder.backend_used,
    full_split_sizes=full_split_sizes, device=str(device), torch=str(torch.__version__),
    dependencies=dependencies, stageb=stageb_info, stageb_backbone=stageb_record, policy_validation=policy_quality,
    recovered_policy=recovery_record, execution_cohorts=budget_reports, policy_training_population="original_train_split_legacy_repaired",
    policy_format="pair_pl_v1", reward_error_scale=rls_cfg["reward_error_scale"])
